In [3]:
import pandas as pd
import json
import re
import anthropic
from groq import Groq
import os
from dotenv import load_dotenv
import numpy as np

load_dotenv()
anthropic_client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

groq_client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

print("all good")

all good


In [ ]:

#generate tweet prompt

def generate_tweets(n):
    prompt = f"""Generate {n} synthetic tweets as a JSON array. Each tweet must conform exactly to this schema:

{{
  "id": "<unique numeric string>",
  "text": "<tweet text, max 280 chars>",
  "created_at": "<ISO 8601 timestamp>",
  "entities": {{
    "urls": [{{"expanded_url": "<full url or empty list>"}}],
    "mentions": [{{"username": "<handle without @>"}}],
    "hashtags": [{{"tag": "<hashtag without #>"}}]
  }},
  "author": {{
    "id": "<unique numeric string>",
    "username": "<twitter handle>",
    "name": "<display name>",
    "verified": <true|false>,
    "created_at": "<ISO 8601 timestamp>",
    "public_metrics": {{
      "followers_count": <int>,
      "following_count": <int>,
      "tweet_count": <int>
    }}
  }},
  "public_metrics": {{
    "retweet_count": <int>,
    "reply_count": <int>,
    "like_count": <int>,
    "quote_count": <int>
  }},
  "label": <1 for scam, 0 for legitimate, -1 for ambiguous>
}}

Generate in this distribution:
- 40% scam (label: 1): crypto giveaway scams, impersonation of Elon Musk/Vitalik/CZ,
  "send ETH get 2x back" schemes, phishing links, urgent limited-time offers.
  Fresh accounts: low followers, high following, young account age, suspicious usernames.
- 40% legitimate (label: 0): real crypto discussion, news sharing, price commentary,
  developer posts. Aged profiles, balanced follower ratios, occasionsally verified.
- 20% ambiguous (label: -1): aggressive but legitimate marketing, new accounts posting
  real content, promotional language that isn't technically a scam.

Make metadata internally consistent — a 3-day-old account should have low tweet_count,
scam tweets should have inflated likes but near-zero replies.
Vary writing style, don't repeat phrases across tweets.
Return ONLY the raw JSON array, no explanation or markdown."""

    return prompt


In [ ]:
#tweets file generation

try: 
    with open("data/tweets.json", "r") as f:
        tweets = json.load(f)
    print(f"loaded {len(tweets)} existing tweets")
except FileNotFoundError:
    tweets = []

for _ in range(10):
    message = anthropic_client.messages.create(
        model="claude-opus-4-5",
        max_tokens=8096,
        messages=[{"role": "user", "content": generate_tweets(10)}]
    )
    response_text = message.content[0].text
    response_text = response_text.strip().removeprefix("```json").removesuffix("```").strip()
    tweets += json.loads(response_text)
    print(f"Total so far: {len(tweets)}")

with open("data/tweets.json", "w") as f:
    json.dump(tweets, f, indent=2)
    

print(f"Final Total: {len(tweets)}")

In [4]:
from snorkel.labeling import labeling_function
import re

SCAM = 1
LEGIT = 0
ABSTAIN = -1


In [5]:
from datetime import datetime, timezone
suspicious_domains = re.compile(
    r"(bit\.ly|tinyurl|t\.co|goo\.gl|ow\.ly|short\.io|rebrand\.ly)"
    )
suspicious_keywords = re.compile(
    r"(giveaway|claim|free|airdrop|crypto|reward|promo|win|bonus)"
    )
@labeling_function()
def lf_giveaway_keywords(x):
    keywords = ["giveaway", "giving back", "send", "get back", "2x", "double" "free"]
    text = x["text"].lower()
    if any(kw in text for kw in keywords):
        return SCAM
    return ABSTAIN

@labeling_function()
def lf_urgency_language(x):
    keywords = ["limited time", "running out", "last chance", "hurry", "act now"]
    text = x['text'].lower()
    if any(kw in text for kw in keywords):
        return SCAM
    return ABSTAIN

@labeling_function()
def lf_new_account(x): 
    followers = int(x["author"]["public_metrics"]["followers_count"])
    created = datetime.fromisoformat(x["author"]["created_at"].replace("Z", "+00:00"))
    age_days = (datetime.now(timezone.utc) - created).days
    tweet_count = int(x['author']['public_metrics']['tweet_count'])

    if (age_days < 30 or tweet_count < 50) and followers < 100: 
        return SCAM
    return ABSTAIN

@labeling_function()
def lf_follower_ratio(x):

    followers_count = x["author"]["public_metrics"]["followers_count"]
    following_count = x["author"]["public_metrics"]["following_count"]

    if following_count > followers_count * 10:
        return SCAM
    if followers_count > followers_count * 100:
        return LEGIT
    return ABSTAIN

@labeling_function()
def lf_crypto_keywords(x):
    text = x['text'].lower()
    crypto_terms = [
        # Bitcoin
        "btc", "bitcoin", "satoshi", "sats",
        # Ethereum
        "eth", "ethereum", "wei", "gwei",
        # Other major coins
        "doge", "dogecoin", "solana", "sol", "xrp", "ripple", "cardano", "ada",
        "litecoin", "ltc", "bnb", "binance", "avax", "avalanche", "matic", "polygon",
        "shib", "shiba", "pepe", "floki"]
    scam_terms = [# Scam-specific crypto language
        "wallet", "blockchain", "crypto", "token", "coin", "defi", "nft",
        "airdrop", "whitelist", "presale", "mint", "claim", "reward",
        "hodl", "moon", "lambo", "ape in", "to the moon"
        ]
    if any(kw in text for kw in crypto_terms) and any(kw in text for kw in scam_terms):
        return SCAM
    return ABSTAIN

@labeling_function()
def lf_impersonation_text(x):
    text = x['text'].lower()
    keywords = ["official", "real", "verified", "ceo"]
    if any(kw in text for kw in keywords):
        return SCAM
    return ABSTAIN

@labeling_function()
def lf_suspicious_url(x):
    urls = x['entities']['urls']
    if not urls:
        return ABSTAIN
    url = urls[0]['expanded_url'].lower()

    if suspicious_domains.search(url) or suspicious_keywords.search(url):
        return SCAM
    return ABSTAIN

@labeling_function()
def lf_engagement_anomaly(x):
    like_count = x['public_metrics']['like_count']
    reply_count = x['public_metrics']['reply_count']

    if like_count > 1000 and reply_count < 5:
        return SCAM
    return ABSTAIN

@labeling_function()
def lf_suspicious_handle(x):
    handle = x['author']['username']
    suspicious_handle_keywords = ['official', 'real', 'verified']
    if any(kw in handle for kw in suspicious_handle_keywords):
        return SCAM
    return ABSTAIN

@labeling_function()
def lf_llm_check(x):
    return x["llm_label"]

@labeling_function()
def lf_verified_legit(x):
    verified = x['author']['verified']
    if verified:
        return LEGIT
    return ABSTAIN


In [6]:
from snorkel.labeling import PandasLFApplier
import time

lfs = [
    lf_giveaway_keywords,
    lf_urgency_language,
    lf_new_account,
    lf_follower_ratio,
    lf_crypto_keywords,
    lf_impersonation_text,
    lf_suspicious_url,
    lf_engagement_anomaly,
    lf_suspicious_handle,
    lf_llm_check,
    lf_verified_legit
]

with open("data/tweets.json", "r") as f:
        tweets = json.load(f)

df = pd.DataFrame(tweets)

def get_llm_label(row):
    time.sleep(0.5)
    tweet = {k: v for k, v in row.items() if k != 'label'}
    prompt = f"""Examine this tweet for crypto scam signals in the text and metadata.
Return only one word: SCAM, LEGIT, or ABSTAIN.
Tweet: {tweet}"""
    
    message = anthropic_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=10,
        messages=[{"role": "user", "content": prompt}]
    )
    response = message.content[0].text.strip().lower()
    if "scam" in response:
        return SCAM
    elif "legit" in response:
        return LEGIT
    return ABSTAIN

df["llm_label"] = df.apply(get_llm_label, axis=1)
df.to_json("data/tweets_with_llm_labels.json", orient="records")

applier = PandasLFApplier(lfs = lfs)

L_train = applier.apply(df)

print(L_train.shape)

100%|██████████| 400/400 [00:00<00:00, 19235.07it/s]

(400, 11)


In [7]:
from snorkel.labeling import LFAnalysis 

LFAnalysis(L = L_train, lfs = lfs).lf_summary()

/Users/yeongbinlee/code/crypto_scam_detector/venv/lib/python3.13/site-packages/snorkel/labeling/analysis.py:61: FutureWarning: Input has data type int64, but the output has been cast to float64.  In the future, the output data type will match the input. To avoid this warning, set the `dtype` parameter to `None` to have the output dtype match the input, or set it to the desired output data type.
  m = sparse.diags(np.ravel(self._L_sparse.max(axis=1).todense()))


,j,Polarity,Coverage,Overlaps,Conflicts
lf_giveaway_keywords,0,[1],0.2350,0.2350,0.0100
lf_urgency_language,1,[1],0.1225,0.1225,0.0000
lf_new_account,2,[1],0.2075,0.1950,0.0175
lf_follower_ratio,3,[1],0.3450,0.3450,0.0050
lf_crypto_keywords,4,[1],0.4175,0.4000,0.0875
lf_impersonation_text,5,[1],0.1825,0.1825,0.0425
lf_suspicious_url,6,[1],0.2950,0.2950,0.0100
lf_engagement_anomaly,7,[1],0.3350,0.3350,0.0000
lf_suspicious_handle,8,[1],0.0375,0.0375,0.0000
lf_llm_check,9,"[0, 1]",0.9150,0.7225,0.1375


In [8]:
from snorkel.labeling.model import LabelModel

label_model = LabelModel(cardinality=2, verbose= True)
label_model.fit(L_train=L_train, n_epochs= 500, lr = 0.001, seed =42)

INFO:root:Computing O...
INFO:root:Estimating \mu...
  0%|          | 0/500 [00:00<?, ?epoch/s]INFO:root:[0 epochs]: TRAIN:[loss=2.689]
INFO:root:[10 epochs]: TRAIN:[loss=2.483]
INFO:root:[20 epochs]: TRAIN:[loss=2.048]
INFO:root:[30 epochs]: TRAIN:[loss=1.497]
INFO:root:[40 epochs]: TRAIN:[loss=0.940]
INFO:root:[50 epochs]: TRAIN:[loss=0.499]
INFO:root:[60 epochs]: TRAIN:[loss=0.248]
INFO:root:[70 epochs]: TRAIN:[loss=0.155]
INFO:root:[80 epochs]: TRAIN:[loss=0.132]
INFO:root:[90 epochs]: TRAIN:[loss=0.122]
INFO:root:[100 epochs]: TRAIN:[loss=0.111]
INFO:root:[110 epochs]: TRAIN:[loss=0.100]
INFO:root:[120 epochs]: TRAIN:[loss=0.090]
INFO:root:[130 epochs]: TRAIN:[loss=0.081]
INFO:root:[140 epochs]: TRAIN:[loss=0.073]
INFO:root:[150 epochs]: TRAIN:[loss=0.065]
INFO:root:[160 epochs]: TRAIN:[loss=0.058]
INFO:root:[170 epochs]: TRAIN:[loss=0.052]
INFO:root:[180 epochs]: TRAIN:[loss=0.046]
INFO:root:[190 epochs]: TRAIN:[loss=0.041]
INFO:root:[200 epochs]: TRAIN:[loss=0.036]
INFO:root:[21

In [10]:
probs = label_model.predict_proba(L = L_train)
df["scam_prob"] = probs[:, 1]
df.to_json("data/tweets_with_probs.json", orient="records", indent=2)


print(df[["text", "label", "scam_prob"]].head(10))
mask = np.max(probs, axis=1) > 0.6

probs_filtered = probs[mask]
df_filtered = df[mask]



print(probs)

                                                text  label  scam_prob
0  🚀 ELON MUSK OFFICIAL GIVEAWAY 🚀\n\nTo celebrat...      1   1.000000
1  ETH gas fees finally reasonable again. Managed...      0   0.060135
2  ⚡ VITALIK BUTERIN ETHEREUM CELEBRATION ⚡\n\nTh...      1   1.000000
3  Just published our Q4 analysis on L2 adoption ...      0   0.000145
4  🔥 NEW GEM ALERT 🔥\n\nJust found $PEPE2.0 befor...     -1   0.981901
5  BTC reclaimed 43k, now testing resistance at 4...      0   0.163378
6  🎁 CZ BINANCE SPECIAL EVENT 🎁\n\nCelebrating Bi...      1   1.000000
7  Finally deployed my first smart contract to ma...      0   0.163378
8  Our token launch is in 48 HOURS! 🚀\n\nWhitelis...     -1   0.946739
9  ⚠️ URGENT - WALLET VERIFICATION REQUIRED ⚠️\n\...      1   1.000000
[[3.03950470e-13 1.00000000e+00]
 [9.39865339e-01 6.01346607e-02]
 [5.32750810e-09 9.99999995e-01]
 [9.99855251e-01 1.44749149e-04]
 [1.80985794e-02 9.81901421e-01]
 [8.36622268e-01 1.63377732e-01]
 [7.98419160e-14 1.00